<a href="https://colab.research.google.com/github/oecdblockchain/Lottery/blob/master/s4/S4_lab_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 4 — Agents and Tool Use
### Letting the model act

**AI Systems in Practice** · HEC Paris · Lab (1h45)

---

You are the AI lead at **Velora**.
In Session 3 the model answered questions from your data.
This week it gets four tools and decides which one to call, in what order, and
when to stop.

> **An agent is a loop in which the model chooses the next action. You hand the model
> control over your execution flow.**

In this lab you will:

1. run an agent with vague tool descriptions and no stopping rule, and measure the cost
   (this part is designed to fail);
2. write better tool descriptions and guardrails, and measure again;
3. run five reference questions and keep the traces;
4. compute what an agent costs: tokens per step, the *n²* effect, €/request;
5. read the traces with a failure taxonomy and find one fact your agent made up.

**Before you start:** `search_web` runs in `cached` mode by default. This is the normal
way to run the lab. No search key is needed.

| | | |
|---|---|---|
| 0. Setup | 10 min | keys, data, four tools |
| A. The loop without guardrails | 15 min | fails on purpose |
| B. Describing the tools | 20 min | TODO 1–2 |
| C. The five runs | 25 min | five traces |
| D. The economics of agency | 15 min | tokens, n², €/request |
| E. Reading the traces | 20 min | failure taxonomy |
| F. Bonus | 10 min | a write tool behind a human gate |

**Deliverable:** this notebook, executed (end of session), plus a one-page report
(23:59 on next Wednesday). See the last cell.


## 0. Setup

One key only: the OpenRouter key from sessions 1 to 3.
It pays for the model **and** for live search, if you use it. If it does not work, raise your hand.


This session needs **tool calling** (also called function calling): the model returns a
structured request `{name, arguments}` instead of text, and your code runs it.
**Not every model supports this.**

The three search modes:

- **`cached`** — about fifty recorded queries served from `search_cache.json`, drawn from
  the twelve pages in `web_snapshot/`. Same output every time. **The default, and the
  mode used for grading.**
- **`live`** — real web search through OpenRouter, on the same key. Results change from
  day to day, calls can fail, nothing is planted. Used in Part F.
- **`replay_fail`** — simulated timeouts and empty results. Used in TODO 4, and in Part A
  if your agent works too well.


**Environment.** One package to install; everything else is standard library plus
`pandas` for the tables.


In [1]:
# Dependencies. `openai` is the only third-party client: OpenRouter speaks the
# OpenAI chat-completions protocol, so one SDK reaches any model on it.
# `pandas` is used for the summary tables only.
!pip install -q openai

import json, os, re, time, zipfile, urllib.request
import collections
from getpass import getpass
from openai import OpenAI
import pandas as pd

print("ready")

ready


**Data download.** All lab files live in one public GitHub folder. Four JSON files and
one zip; nothing else is fetched from the network in `cached` mode.


In [2]:
# Download the lab data into the working directory: four JSON files and one zip
# (the twelve markdown pages behind the cached searches). Nothing else is fetched
# from the network in `cached` mode.
# All files for this lab are in one public folder.
BASE = "https://raw.githubusercontent.com/vorpax/velora-labs/refs/heads/main/s4/"

FILES   = [
           "orders_db.json",
           "questions_dev.json",
           "kb_snippets.json",
           "search_cache.json",
]
FOLDERS = ["web_snapshot"]      # shipped as a .zip, unpacked here

# One HTTP GET per file; the zip is unpacked in place.
for name in FILES:
    urllib.request.urlretrieve(BASE + name, name)
for name in FOLDERS:
    urllib.request.urlretrieve(BASE + name + ".zip", name + ".zip")
    zipfile.ZipFile(name + ".zip").extractall()

print("downloaded:", sorted(p for p in os.listdir() if not p.startswith(".")))

downloaded: ['kb_snippets.json', 'orders_db.json', 'questions_dev.json', 'sample_data', 'search_cache.json', 'web_snapshot', 'web_snapshot.zip']


**Loading the data.** Four in-memory objects and two question splits. `web_snapshot/`
is reference material only: the recorded searches were built from it, and no cell reads
it.


In [3]:
# Load the datasets into memory and split the questions by role:
# REFERENCE (5, run in Part C) and AB (4, used for the Part A vs Part B comparison).
ORDERS    = json.load(open("orders_db.json"))       # load 60 Velora orders from json file
QUESTIONS = json.load(open("questions_dev.json"))   # load 9 questions: 5 reference + 4 for the A/B test from json
KB        = json.load(open("kb_snippets.json"))     # load the Session 3 documents from json file (the knowledge base)
CACHE     = json.load(open("search_cache.json"))    # load ~50 mock web searches
# web_snapshot/ holds the 12 pages (in markdown format, not in real html) the recorded searches come from. No cell reads it.

# Index by question id; used by the pivot, TODO 4 and the bonus cells.
BY_QID = {q["id"]: q for q in QUESTIONS}
REFERENCE = [q for q in QUESTIONS if q["split"] == "reference"]
AB        = [q for q in QUESTIONS if q["split"] == "ab"]

print(f"{len(ORDERS)} orders, {len(QUESTIONS)} questions "
      f"({len(REFERENCE)} reference + {len(AB)} for the A/B), "
      f"{len(KB)} knowledge-base snippets, {len(CACHE['entries'])} cached searches")
print()

# min_steps = fewest steps a competent analyst would need.
# tools_required = the tools that actually hold the answer.
for q in REFERENCE:
    print(f"  {q['id']}  min_steps={q['min_steps']}  tools={q['tools_required']}")
    print(f"       {q['question'][:96]}")

60 orders, 9 questions (5 reference + 4 for the A/B), 10 knowledge-base snippets, 50 cached searches

  Q1  min_steps=3  tools=['query_orders', 'search_web', 'calculate']
       Across the orders shipped to the United Kingdom in March 2026 and billed in pounds, what was the
  Q2  min_steps=4  tools=['query_orders', 'search_web', 'calculate']
       What was the gross margin, in euros and as a percentage, on the euro-denominated Cargo Pro order
  Q3  min_steps=2  tools=['search_kb']
       A customer bought a Trail X 14 months ago and the battery has stopped holding charge. The batter
  Q4  min_steps=1  tools=[]
       Which company manufactures the battery cells used in the Velora Trail X, and what is the cell ch
  Q5  min_steps=2  tools=['query_orders']
       Julia Weber is asking about her Cargo Pro order. What did she pay, and what is its status?


**Model.** One configuration dictionary: provider URL, model id, and the prices used
for every cost figure in this notebook.


In [4]:
# Model configuration: provider URL, model id, and prices in EUR per million tokens.
# The prices feed every cost figure in this notebook (see summarise_run).
# One provider for the whole course: OpenRouter, through the OpenAI SDK.
CFG = {
    "base_url": "https://openrouter.ai/api/v1",
    "model": "z-ai/glm-5.3-flash",   # GLM 5.3-flash is tailored for agentic work/reasoning.
    "price_in": 0.075, "price_out": 0.25,      # EUR per 1M tokens
}
MODEL = CFG["model"]

print(f"{MODEL} via OpenRouter, {CFG['price_in']} EUR / 1M input tokens")

z-ai/glm-5.3-flash via OpenRouter, 0.075 EUR / 1M input tokens


**Client.** The OpenAI SDK pointed at OpenRouter. Retries matter more for an agent than
for a single call: a run is a chain, and one dropped response breaks the chain.


In [5]:
# Build the API client. The key is read interactively so it is never stored in the notebook.
# An agent run is a chain of calls: one transient failure must not abort the whole chain.
# The client. `max_retries` retries on a 429 (too much requests) or 5xx error (internal server error).
# An agent makes one call per step, so one failed response would otherwise kill the whole run.

API_KEY = getpass("OpenRouter API key: ").strip() # ask for openrouter api key.

client = OpenAI(api_key=API_KEY, base_url=CFG["base_url"], max_retries=5) # initialize the OpenAI client for OpenRouter.

print(f"client ready — {MODEL} at {CFG['base_url']}")

OpenRouter API key: ··········
client ready — z-ai/glm-5.3-flash at https://openrouter.ai/api/v1


**Search mode.** One variable controls where `search_web` looks. Every later cell reads
it.


In [6]:
# Select where search_web gets its results. Only SEARCH_MODE is meant to change.
# cached = deterministic (needed for grading), live = real and non-reproducible,
# replay_fail = failures injected on purpose.
# ----------------------------------------------------------- THE SEARCH MODE
# "cached"      served from search_cache.json. Same output every time. NO EXTRA KEY.
#               The default, and a choice rather than a fallback: the defects this
#               lab asks you to find are in the recording, not on the web.
# "live"        real web search, through OpenRouter https://openrouter.ai/docs/guides/features/server-tools/web-search.
# "replay_fail" simulated timeouts and empty results, for TODO 4 and the Part A pivot.

SEARCH_MODE = "cached"       # <- change this line only

# Live search runs as a second, cheap completion whose only job is to trigger
# OpenRouter's server-side search. What comes back to you is the citation list from the search
# In live mode a second, cheap model triggers OpenRouter's server-side search.
# Only the citations it returns are kept; its text is discarded.
SEARCH_MODEL = "deepseek/deepseek-v4-flash-0731"

print(f"search : {SEARCH_MODE}")
if SEARCH_MODE == "live":
    print(f"         {SEARCH_MODEL} + openrouter:web_search. Billed per search.")
else:
    print("         Latencies from search are REPLAYED, not measured. Say so in your report.")

search : cached
         Latencies from search are REPLAYED, not measured. Say so in your report.


### The four tools

The agent sees four names, four descriptions and four argument schemas. That is its
entire view of the world.

| tool | what it reaches | note |
|---|---|---|
| `query_orders` | the 60-order Velora database | filters rows, does not sum |
| `search_kb` | the Session 3 documents | your own documents |
| `search_web` | the outside world | the only tool with unpredictable results |
| `calculate` | a Python expression | because the model is not reliable at arithmetic |

`calculate` does not make the model better at arithmetic. It lets the model choose not
to do arithmetic itself. Whether it makes that choice is visible in the trace.


In [7]:
# Tool 1 of 4: query_orders. Every argument is an optional filter; filters combine with AND.
# Output: {"count": n, "orders": [...]} capped at 40 rows.
# First tool. It FILTERS the orders and returns the matching rows.

# No aggregation on purpose: a sum must go through calculate(), which makes the arithmetic visible in the trace.

def query_orders(product=None, country=None, currency=None, status=None,
                 month=None, customer_name=None, order_id=None):
    """Filter the order table. Returns matching rows, never an aggregate."""
    rows = list(ORDERS.values())
    if order_id:      rows = [r for r in rows if r["order_id"] == order_id] # if order_id is given
    if product:       rows = [r for r in rows if r["product"] and product.lower() in r["product"].lower()] # then filter by product, country, currency, status, month, customer_name
    if country:       rows = [r for r in rows if r["country"] == country.upper()]
    if currency:      rows = [r for r in rows if r["currency"] == currency.upper()]
    if status:        rows = [r for r in rows if r["status"] == status.lower()]
    if month:         rows = [r for r in rows if r["order_date"].startswith(month)]
    if customer_name: rows = [r for r in rows if customer_name.lower() in r["customer_name"].lower()]
    return {"count": len(rows), "orders": rows[:40]}

# Smoke test: two known lookups.
print("GB/GBP March orders:", query_orders(country="GB", currency="GBP", month="2026-03")["count"])
print("one order          :", query_orders(order_id="VLR-2026-04640")["orders"][0]["customer_name"])

GB/GBP March orders: 4
one order          : Oliver Bennett


In [8]:
# Tool 2 of 4. a mockup of Session 3's retriever, now a tool. Keyword scoring on title + section + text, top-k (default 3), three documents max,
# and nothing if nothing matches.

def search_kb(query, k=3):
    """Keyword search over the Velora knowledge base."""
    # Tokenise the query: lowercase, alphanumerics only, drop tokens of 1-2 characters.
    terms = [t for t in re.sub(r"[^a-z0-9 ]", " ", query.lower()).split() if len(t) > 2]
    scored = []
    # Score = total number of term occurrences in the document.
    for d in KB:
        hay = (d["title"] + " " + d["section"] + " " + d["text"]).lower()
        scored.append((sum(hay.count(t) for t in terms), d))
    scored.sort(key=lambda x: -x[0])
    return {"results": [{"doc_id": d["doc_id"], "title": d["title"],
                         "section": d["section"], "text": d["text"]}
                        for s, d in scored[:k] if s > 0]}

print("warranty  :", [r["doc_id"] for r in search_kb("battery warranty cycles")["results"]])
print("nonsense  :", search_kb("zzzz nonexistent term")["results"])

warranty  : ['KB-WAR-01', 'KB-OPS-01', 'KB-WAR-02']
nonsense  : []


In [9]:
# Tool 3 of 4: calculate. Evaluates one arithmetic expression with eval().
# Safety relies on the regex whitelist: digits, . + - * / ( ) whitespace , and %.
# Would otherwise allow to evaluate arbitrary code, e.g. __import__('os').system('ls'). (list files in the current directory)

def calculate(expression):
    """Evaluate one arithmetic expression. No names, no calls, no attribute access."""
    if not re.fullmatch(r"[0-9\.\+\-\*/\(\)\s,%]+", expression or ""):
        return {"error": "unsupported_expression",
                "detail": "digits, + - * / ( ) and . only"}
    # Thousands separators are stripped. Builtins are removed from the eval
    # namespace as a second layer.
    try:
        return {"expression": expression,
                "result": eval(expression.replace(",", ""), {"__builtins__": {}}, {})}
    except Exception as e:
        return {"error": type(e).__name__, "detail": str(e)[:120]}

print("arithmetic:", calculate("6099 / 0.855"))
print("refused   :", calculate("__import__('os').system('ls')"))

arithmetic: {'expression': '6099 / 0.855', 'result': 7133.333333333333}
refused   : {'error': 'unsupported_expression', 'detail': 'digits, + - * / ( ) and . only'}


**Cache lookup.** `cached` mode is a dictionary lookup. The key is a normalised form of
the query, so the cache hits on wording variants of a recorded query and misses on
everything else. A miss looks exactly like an empty web result.


In [10]:
# Cache lookup key for `cached` mode. Two differently worded queries map to the same
# key when they share the same content words (normalized, not using raw query).

STOPWORDS = set(CACHE["stopwords"])

def normalise_query(q):
    """Must match build_dataset.py's norm_query exactly, or the cache never hits."""
    q = re.sub(r"[^a-z0-9 ]+", " ", str(q).lower())
    return " ".join(sorted({t for t in q.split() if t and t not in STOPWORDS}))

# Queue consumed by `replay_fail` mode, one entry per search_web call.
# An empty queue means "timeout" for every remaining call.
FAIL_PLAN = []      # replay_fail pops from here: "timeout", "empty", or "ok"

print(normalise_query("What is the average GBP to EUR exchange rate in March 2026?"))
print(normalise_query("march 2026 gbp eur exchange rate"))
print("both keys are in the cache:",
      normalise_query("GBP to EUR exchange rate March 2026") in CACHE["entries"])

2026 average eur exchange gbp march rate
2026 eur exchange gbp march rate
both keys are in the cache: True


### `search_web`, tool 4 of 4

The only tool that leaves the machine, so the only one that can be slow, fail, or return
different results each time.
The replay layer in front of it makes grading reproducible.

All three modes return the same shape: `{"results": [{"title", "url", "published",
"snippet"}, ...]}`. Only the `source` field says where the results came from.


In [13]:
# Tool 4 of 4: search_web. The only tool that can be slow, fail, or change between runs.
# Three branches, one output shape:
#   {"results": [{title, url, published, snippet}, ...], "latency_s": ..., "source": ...}
# Branch order: replay_fail -> live -> cached (cached is also the fallback).
def search_web(query, max_results=3):
    """Search the web. Honours SEARCH_MODE. Same result shape in all three modes."""
    t0 = time.time()

    # 1. Simulated failures. "timeout" returns an error dict, "empty" returns
    #    zero results, "ok" falls through to the mode below.
    if SEARCH_MODE == "replay_fail":
        mode = FAIL_PLAN.pop(0) if FAIL_PLAN else "timeout"
        if mode == "timeout":
            time.sleep(0.2)
            return {"error": "timeout", "detail": "search provider did not respond in 10s",
                    "latency_s": round(time.time() - t0, 3), "source": "replay_fail"}
        if mode == "empty":
            return {"results": [], "latency_s": round(time.time() - t0, 3),
                    "source": "replay_fail"}

    # 2. Real search. The completion is only a trigger; the citations in `annotations` is what we are looking for.
    if SEARCH_MODE == "live":
        # The search runs on OpenRouter's side. This model is not asked to report what
        # it found: it is asked to search. What we keep are the citations it came back
        # with, so no second model stands between the web and your agent.
        try:
            r = client.chat.completions.create(
                model=SEARCH_MODEL,
                messages=[{"role": "user",
                           "content": f"Search the web for: {query}\nReply with one word: done."}],
                tools=[{"type": "openrouter:web_search",
                        "parameters": {"engine": "exa", "max_results": max_results}}])
            msg = r.model_dump()["choices"][0]["message"]
            out = []
            for a in (msg.get("annotations") or []):
                c = a.get("url_citation") or a
                if not c.get("url"):
                    continue
                out.append({
                    "title": c.get("title") or c["url"],
                    "url": c["url"],
                    # Live citations usually carry no date. In `cached` every result
                    # has one — which is why the stale-rate trap only works there.
                    "published": (c.get("published") or c.get("published_date")
                                  or c.get("date") or "unknown"),
                    "snippet": (c.get("content") or c.get("snippet") or "")[:520]})
            return {"results": out[:max_results],
                    "latency_s": round(time.time() - t0, 3),   # model + search, not search
                    "source": "live",
                    **({"note": "no citations returned"} if not out else {})}
        except Exception as e:
            return {"error": type(e).__name__, "detail": str(e)[:120],
                    "latency_s": round(time.time() - t0, 3), "source": "live"}

    # cached mode (also the fallback for everything else)
    # 3. Recording. A miss returns [] with a note: same shape as a real empty result.
    entry = CACHE["entries"].get(normalise_query(query))
    if entry is None:
        return {"results": [], "note": "no cached result for this query",
                "latency_s": round(time.time() - t0, 3), "source": "cached"}
    return {"results": [{k: r[k] for k in ("title", "url", "published", "snippet")}
                        for r in entry["results"][:max_results]],
            "latency_s": round(time.time() - t0, 3), "source": "cached"}

In [14]:
# Probe: one cached search, showing the fields the agent will see.
# Note the two dates: one result is on-period, the other is six months old.
# A search result: a title, a URL, a date, and 520 characters of text.
# The page is in web_snapshot/.
probe = search_web("GBP to EUR exchange rate March 2026")
print(f"search_web [{probe['source']}] -> {len(probe['results'])} results")
for r in probe["results"]:
    print(f"  {r['published']}  {r['title']}")
    print(f"     {r['url']}")

search_web [cached] -> 2 results
  2026-04-01  Reference exchange rates — March 2026
     https://ecb-reference.example.org/fx-rates-2026-03
  2025-10-01  Reference exchange rates — September 2025
     https://ecb-reference.example.org/fx-rates-2025-09


**Look at the `published` field.** Every cached result carries a date.
It is your only defence against a figure that is correct but from the wrong month. Whether your agent
reads that field is visible in the trace.

In `live` mode most citations would come back with `published: "unknown"`. The date is a
property of the recording, not of the web.


## A. The loop without guardrails · 15 min

The whole agent:

```
messages = [system, question]
while True:
    reply = model(messages, tools)          # the model chooses
    if no tool call:  return reply          # ... including when to stop
    result = dispatch(reply.tool_call)      # YOUR code executes
    messages.append(result)                 # and everything goes back in
```

Four consequences:

1. **The model chooses the next call.** Your code only runs what it asked for.
2. **The model chooses when to stop.** `max_steps` caps the budget. It does not
   guarantee a correct answer.
3. **Everything accumulates.** The full history is re-sent at every step: Session 2's
   *n²* formula, with a much larger constant.
4. **Tool output enters the prompt.** A web page your agent reads is a prompt written
   by a stranger.

This first version has vague tool descriptions and no stopping rule. It is designed to
fail: not to crash, but to burn steps and money while producing answers that look fine.
Do not fix it. Measure it.


### `execute_tool_call` — where your code acts

The model emits a tool name and a JSON string of arguments. This function looks the name
up in a table you wrote, calls the Python function you wrote, and returns the result as a
`tool` message. The model asked. Your code decided to run it.


In [ ]:
# Dispatch table and audit log. DISPATCH maps a tool name (a string chosen by the model)
# to a Python function (written by you). execute_tool_call() is the only place where a
# model request becomes an action.
DISPATCH = {"query_orders": query_orders, "search_kb": search_kb,
            "search_web": search_web, "calculate": calculate}

TOOL_LOG = []      # every call, in order, with its result. This is your audit trail.

# The step cap is enforced in the loop below. TODO 2 is about what the agent should
# do when it reaches it.
HARD_CAP = 20

def execute_tool_call(step, tc, entry, verbose):
    """Run ONE tool call the model asked for, record it, return the tool message.

    The model asked. This function — your code — decided to honour it.
    """
    # Parse the request. Malformed JSON arguments degrade to {} instead of crashing the run.
    name = tc.function.name
    try:
        args = json.loads(tc.function.arguments or "{}")
    except Exception:
        args = {}
    # An unknown tool name is reported back to the model, not raised.
    fn = DISPATCH.get(name)                       # a table YOU wrote
    result = fn(**args) if fn else {"error": "unknown_tool", "detail": f"{name} is not a tool"}
    # Two records: TOOL_LOG (flat, session-wide, for the audit) and
    # entry["calls"] (per step, for the trace).
    TOOL_LOG.append({"step": step, "tool": name, "args": args,
                     "ok": "error" not in str(result)[:200]})
    entry["calls"].append({"tool": name, "args": args,
                           "result": json.dumps(result, ensure_ascii=False)[:900]})
    if verbose:
        print(f"[{step}] {name}({json.dumps(args, ensure_ascii=False)[:100]})")
        print(f"     -> {json.dumps(result, ensure_ascii=False)[:160]}")
    # The tool message carries the id of the model's tool_call. Content is capped
    # at 4000 characters before it enters the prompt.
    return {"role": "tool", "tool_call_id": tc.id,
            "content": json.dumps(result, ensure_ascii=False)[:4000]}

print(f"{len(DISPATCH)} tools reachable, hard cap {HARD_CAP} steps")

### `to_message` — the protocol

The `tool_calls` in the assistant turn must go back exactly as received: same ids, same
argument strings. A `tool` message with an id the provider never issued makes it reject
the whole request. This is the most common way a hand-written agent loop fails on its
second step.


In [ ]:
# Convert the SDK response object back into a plain dict for the `messages` list.
# tool_calls must be formated exactly (id, name, argument string): the provider matches
# every later `tool` message against these ids.
def to_message(m):
    """Rebuild the assistant turn so it can go back into `messages`."""
    msg = {"role": "assistant", "content": m.content or ""}
    if m.tool_calls:
        msg["tool_calls"] = [{"id": tc.id, "type": "function",
                              "function": {"name": tc.function.name,
                                           "arguments": tc.function.arguments}}
                             for tc in m.tool_calls]
    return msg

print("the ids in to_message() must match the tool_call_id in execute_tool_call()")

**Bookkeeping.** Two helpers that build the trace and the run summary. They contain no
agent logic.


In [ ]:
# Trace and summary builders. record_step() opens one entry per model call;
# summarise_run() aggregates tokens, cost, latency and the list of tools used.
# cost_eur is computed from the prices in CFG.
# These two functions build the records the loop returns. `trace`
# is built even when the run fails.
def record_step(step, m, dt, usage):
    """One step of the trace, before its tool results are attached."""
    return {"step": step, "thought": m.content or "", "latency_s": round(dt, 3),
            "in_tokens": usage.prompt_tokens, "out_tokens": usage.completion_tokens,
            "calls": []}

def summarise_run(trace, in_tok, out_tok, t_start, stop_reason):
    """Everything a run produces besides its answer: steps, cost, latency, tools."""
    return {"steps": len(trace), "stop_reason": stop_reason, "trace": trace,
            "in_tokens": in_tok, "out_tokens": out_tok,
            "latency_s": round(time.time() - t_start, 3),
            "cost_eur": in_tok * CFG["price_in"] / 1e6 + out_tok * CFG["price_out"] / 1e6,
            "tools_used": [c["tool"] for s in trace for c in s["calls"]]}

print("bookkeeping ready")

### `run_agent` — the loop

Per step: call the model, count tokens, append its reply to the history, check whether
it called a tool, run each tool call, append the results. A reply with no tool call is
the answer. A cap hit returns `None`.


In [ ]:
# Arguments: the question, the tool schemas (what the model sees), the system prompt,
# a step cap, and temperature (0.0 = as deterministic as the provider allows).
# Returns (answer, summary). answer is None when the cap is hit.
# ============================ THE LOOP ========================================
# This is the whole idea of the session. Read every line.

def run_agent(question, tools, system_prompt, max_steps=8, temperature=0.0, verbose=True):
    """One ReAct-style loop. Returns (final_answer, summary); summary carries the trace."""
    max_steps = min(max_steps, HARD_CAP)
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user", "content": question}]
    trace, in_tok, out_tok, t_start = [], 0, 0, time.time()

    # One iteration = one model call = one step. Each call re-sends the full
    # `messages` list, so the input grows at every step.
    for step in range(1, max_steps + 1):
        t0 = time.time()
        r = client.chat.completions.create(
            model=MODEL, temperature=temperature, messages=messages,
            tools=tools, tool_choice="auto")
        dt = time.time() - t0
        m = r.choices[0].message
        in_tok += r.usage.prompt_tokens
        out_tok += r.usage.completion_tokens

        entry = record_step(step, m, dt, r.usage)
        messages.append(to_message(m))    # the history grows at every step

        # Stop condition: a reply with no tool call is treated as the final answer.
        if not m.tool_calls:                      # THE MODEL DECIDED TO STOP.
            entry["final"] = m.content            # Not your code.
            trace.append(entry)
            if verbose: print(f"[{step}] FINAL: {(m.content or '')[:150]}")
            return m.content, summarise_run(trace, in_tok, out_tok, t_start, "stopped")

        # The model may emit several tool calls in one step. Each is executed and
        # its result appended before the next model call.
        for tc in m.tool_calls:                   # YOUR code executes, via DISPATCH
            messages.append(execute_tool_call(step, tc, entry, verbose))
        trace.append(entry)

    # The step budget ran out. The agent did not fail loudly. It just stopped.
    # TODO 2 is about what happens here.
    if verbose: print(f"[!] hit max_steps={max_steps} without a final answer")
    return None, summarise_run(trace, in_tok, out_tok, t_start, "max_steps")

print("run_agent ready — that is the entire agent, and there is nothing else to it.")

### Version 0 — vague descriptions

Read the four descriptions and predict what goes wrong before running the next cell.
Three of them could describe `search_web`.


In [ ]:
# Version 0 tool schemas. Same four functions, same parameters, but descriptions of
# two to four words. The model has nothing else to choose between them with.
# ----------------------------------- version 0: vague descriptions, no stopping rule
# Read these descriptions and predict what will go wrong BEFORE you run the cell.
TOOLS_VAGUE = [
    {"type": "function", "function": {
        "name": "search_web", "description": "Search for any information.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "query_orders", "description": "Get order data.",
        "parameters": {"type": "object", "properties": {
            "product": {"type": "string"}, "country": {"type": "string"},
            "currency": {"type": "string"}, "status": {"type": "string"},
            "month": {"type": "string"}, "customer_name": {"type": "string"},
            "order_id": {"type": "string"}}}}},
    {"type": "function", "function": {
        "name": "search_kb", "description": "Look things up.",
        "parameters": {"type": "object",
                       "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {
        "name": "calculate", "description": "Do maths.",
        "parameters": {"type": "object",
                       "properties": {"expression": {"type": "string"}},
                       "required": ["expression"]}}},
]

SYSTEM_VAGUE = "You are a helpful assistant for Velora. Use the tools to answer."

print(sum(len(t["function"]["description"].split()) for t in TOOLS_VAGUE),
      "words of tool description. Which tool would YOU pick for a question about",
      "an order paid in pounds?")

Run the four A/B questions with version 0. Watch the step counts, not the answers.


In [ ]:
# Part A run: the four A/B questions with the vague tools. The cap is raised to HARD_CAP
# so the loop can fail visibly. runs_a keeps the full summary (trace included) per question.
# The four A/B questions, run with the vague toolset. Watch the step counts.
TOOL_LOG.clear()
runs_a = {}
for q in AB:
    print("=" * 78); print(f"{q['id']}  (floor: {q['min_steps']} steps)  {q['question'][:70]}")
    ans, s = run_agent(q["question"], TOOLS_VAGUE, SYSTEM_VAGUE, max_steps=HARD_CAP)
    runs_a[q["id"]] = {**s, "answer": ans, "qid": q["id"], "min_steps": q["min_steps"]}

# Summary table without the trace field. excess_steps = steps taken minus the floor.
rows = []
for r in runs_a.values():
    rows.append({k: v for k, v in r.items() if k != "trace"})   # the trace is too big for a table
A = pd.DataFrame(rows)
A["excess_steps"] = A.steps - A.min_steps
print("\n" + "=" * 78)
print(A[["qid", "min_steps", "steps", "excess_steps", "stop_reason",
         "in_tokens", "cost_eur", "latency_s"]].to_string(index=False))
print(f"\nMEAN STEPS          : {A.steps.mean():.1f}   (floor: {A.min_steps.mean():.1f})")
print(f"NON-TERMINATING     : {(A.stop_reason == 'max_steps').sum()} of {len(A)}")
print(f"MEAN EUR PER REQUEST: {A.cost_eur.mean():.5f}")
print(f"AT 400 REQ/DAY      : {A.cost_eur.mean() * 400 * 365:.0f} EUR/year   <- number 1 for your report")

**What to look at.** Not whether the answers are right. Three numbers:

- **mean steps against the floor.** `min_steps` is what a competent analyst would need.
- **how many runs hit `max_steps`.** Look at what the agent returned: `None`, or a
  plausible partial answer.
- **the euros.** One request is cheap. 400 or even 4000 are not.

**Why it fails.** `"Search for any information"` is true of three tools out of four.
The model picks by wording, not by fit, and goes to the web for a question whose answer is in
your own database.
With no stopping rule, it keeps looking for confirmation of what it already has.

*If your agent did well*, set `SEARCH_MODE` to `"replay_fail"` and run the pivot cell
below. The question becomes: what does your agent do when a tool dies?


### The pivot — a dead tool

Same question (Q7), same vague tools, but `search_web` now times out or returns nothing
on a fixed schedule. The output to read is not the step count but whether the final
answer mentions the failure.


In [ ]:
# Pivot: same question (Q7), same vague tools, but search_web now fails on a fixed schedule.
# FAIL_PLAN is consumed in order; once empty, every call times out. SEARCH_MODE is
# restored afterwards so the rest of the notebook is unaffected.
# ------------------------------------------------- the pivot: what happens when a tool dies
# FAIL_PLAN is consumed one entry per search_web call.
_saved_mode = SEARCH_MODE
SEARCH_MODE = "replay_fail"
FAIL_PLAN.clear()
FAIL_PLAN.extend(["timeout", "timeout", "empty", "timeout", "empty", "timeout"])

print("Q7 with a dead search tool. Watch what it does with the exchange rate.\n")
ans_fail, s_fail = run_agent(BY_QID["Q7"]["question"], TOOLS_VAGUE, SYSTEM_VAGUE, max_steps=10)

SEARCH_MODE = _saved_mode
print("\n" + "=" * 78)
print("ANSWER:", ans_fail)
print(f"steps={s_fail['steps']}  stop_reason={s_fail['stop_reason']}")
print()
print("The two questions to answer in your report:")
print("  1. Did it report the failure, or did it invent an exchange rate and carry on?")
print("  2. If you were handed only the final answer, would you know a tool had died?")

## B. Describing the tools · 20 min

Tool descriptions are the only information the model has when it picks a tool. It never
sees the Python behind the name. Rewriting the descriptions is the cheapest way to steer
tool choice.


### TODO 1 — rewrite the four tool descriptions

A description answers one question: **when should the model use this tool rather than
the other three?**

State:

- what the tool knows about, and what it does **not** know about;
- when to prefer it over its closest neighbour, by name;
- what it returns and what it does not do (`query_orders` filters; it does not sum);
- the format of an argument where the model would otherwise guess (look at how
  `query_orders` compares `country` and `month`).

Every word is re-sent at every step. Write the shortest text that removes the confusion
measured in Part A.


In [ ]:
# TODO 1 workspace. Each schema is its own variable so one description can be edited and
# re-run. Function names and parameter names must stay identical: they are the keys of
# DISPATCH and the keyword arguments of the Python functions.
SEARCH_WEB = {"type": "function", "function": {
        "name": "search_web",
        "description": "...",     # <- when the web, rather than search_kb?
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "..."}},
            "required": ["query"]}}}


QUERY_ORDERS = {"type": "function", "function": {
        "name": "query_orders",
        "description": "...",     # <- what does it return? what does it NOT do?
        "parameters": {"type": "object", "properties": {
            "product":  {"type": "string", "description": "..."}, # <- describe what the product parameter is
            "country":  {"type": "string", "description": "..."}, # <- describe what the country parameter is
            "currency": {"type": "string", "description": "..."}, # <- describe what the currency parameter is
            "status":   {"type": "string", "description": "..."}, # <- describe what the status parameter is
            "month":    {"type": "string", "description": "..."}, # <- describe what the month parameter is
            "customer_name": {"type": "string", "description": "..."}, # <- describe what the customer_name parameter is
            "order_id": {"type": "string", "description": "..."}}}}} # <- describe what the order_id parameter is

SEARCH_KB = {"type": "function", "function": {
        "name": "search_kb",
        "description": "...",     # <- what is IN the knowledge base?
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string", "description": "..."}}, # <- describe what the query parameter is (regarding the knowledge base search)
            "required": ["query"]}}}

CALCULATE = {"type": "function", "function": {
        "name": "calculate",
        "description": "...",     # <- why would a model that "can do maths" call this?
        "parameters": {"type": "object", "properties": {
            "expression": {"type": "string", "description": "..."}}, # <- describe what the expression parameter is (regarding the calculation tool)
            "required": ["expression"]}}}


TOOLS_V2 = [
    SEARCH_WEB,
    QUERY_ORDERS,
    SEARCH_KB,
    CALCULATE,
]

# Sanity checks (currently commented out) and the description word count,
# which is re-sent on every step.
_descs = [t["function"]["description"] for t in TOOLS_V2]
assert all(len(d) > 60 for d in _descs), "You should write a description for each tool that is at least 60 characters long."
assert len(set(_descs)) == 4, "two tools carry the same description :("
print(f"{sum(len(d.split()) for d in _descs)} words of tool description, "
      f"re-sent on every step of every run.")

### TODO 2 — the guardrails

The hard cap of 20 steps is enforced for you. Yours is the **policy**: what the agent
does near the limit, and what the system returns when it hits it.

An agent stopped at step 8 does not fail loudly. It returns a plausible partial answer,
indistinguishable from a complete one. A higher cap does not fix this. Only a completion
check in code does.

| | |
|---|---|
| `STEP_BUDGET` | an integer, justified by the `excess_steps` measured in Part A |
| `BUDGET_POLICY` | what the agent is **told** to do near the limit — prompt |
| `INCOMPLETE_CONTRACT` | what your **system** returns when the cap is hit — code |


In [ ]:
# TODO 2 workspace. STEP_BUDGET is passed as max_steps in Part C. BUDGET_POLICY is text
# appended to the system prompt (a request to the model). INCOMPLETE_CONTRACT is what the
# caller should receive on truncation (a rule for your code; not yet wired into run_agent).
STEP_BUDGET = 5            # <- an integer
BUDGET_POLICY = ""          # <- a string, added to the system prompt
INCOMPLETE_CONTRACT = ""    # <- a string: what the caller receives on truncation

assert isinstance(STEP_BUDGET, int) and 1 <= STEP_BUDGET <= HARD_CAP
assert all(isinstance(x, str) and len(x) > 60
           for x in (BUDGET_POLICY, INCOMPLETE_CONTRACT))
print(f"budget {STEP_BUDGET} steps")
print("policy   :", BUDGET_POLICY)
print("contract :", INCOMPLETE_CONTRACT)

**System prompt, version 2.** Fixed rules plus your policy. A rule in the prompt is a
request to the model; a rule in code is enforced.


In [ ]:
# System prompt for version 2: fixed rules (sources of truth, arithmetic, dates, refusal)
# plus your BUDGET_POLICY. None of these lines is enforced by code.
# The system prompt for version 2. Your BUDGET_POLICY is added at the end.
# A policy in a prompt is a request, not a rule.
SYSTEM_V2 = f"""You are Velora's internal analyst assistant. You answer questions about
orders, margins, policy and market context by calling tools.

Rules.
- Velora's order database is the only source of truth for what a customer paid.
- Velora's knowledge base is authoritative for Velora policy.
- Use calculate for every arithmetic step, including easy ones.
- Check the `published` date on any web result before you use a figure from it.
- If the information required does not exist in any tool, say so plainly. Do not
  estimate, and do not substitute a plausible figure.

{BUDGET_POLICY}"""

print(SYSTEM_V2)

*Collective point.* Three pairs read out their A→B step counts. The only thing that
differs between them is the text written in TODO 1.


## C. The five runs · 25 min

Five reference questions, your tools, your guardrails. Keep every trace: they are the
material for Part E.

One of the five questions has an answer that exists nowhere: not in the orders, not in
the knowledge base, not on the web. Do not try to guess which one. Run all five, then
read what your agent did.


In [ ]:
# Part C: the five reference questions with TOOLS_V2, SYSTEM_V2 and STEP_BUDGET.
# runs_c keeps every trace; Part E reads them.
TOOL_LOG.clear()
runs_c = {}
for q in REFERENCE:
    print("=" * 78)
    print(f"{q['id']}  floor={q['min_steps']}  tools_needed={q['tools_required'] or 'none'}")
    print(q["question"])
    print("-" * 78)
    ans, s = run_agent(q["question"], TOOLS_V2, SYSTEM_V2, max_steps=STEP_BUDGET)
    runs_c[q["id"]] = {**s, "answer": ans, "qid": q["id"], "min_steps": q["min_steps"],
                       "tools_required": q["tools_required"]}
    print(f"\n  -> {str(ans)[:400]}")

# Summary table. excess = steps minus the floor; n_calls counts tool calls
# (several per step are possible).
rows = []
for r in runs_c.values():
    rows.append({k: v for k, v in r.items() if k != "trace"})
C = pd.DataFrame(rows)
C["excess"] = C.steps - C.min_steps
C["n_calls"] = C.tools_used.apply(len)
display(C[["qid", "min_steps", "steps", "excess", "n_calls", "stop_reason",
           "in_tokens", "out_tokens", "cost_eur", "latency_s"]].round(5))

**Tools used vs tools needed.** A first check before reading the traces.


In [ ]:
# Tool usage versus tool need, per question. unnecessary = called but not needed;
# never_called = needed but not called.
# Tools used by each question, compared to the tools it needed.
rows = []
for qid, r in runs_c.items():
    used = collections.Counter(r["tools_used"])
    need = set(r["tools_required"])
    rows.append({"qid": qid, "needed": ", ".join(sorted(need)) or "-",
                 "question": BY_QID[qid]["question"],
                 "used": ", ".join(f"{k} x {v}" for k, v in sorted(used.items())) or "-",
                 "unnecessary": ", ".join(sorted(set(used) - need)) or "-",
                 "never_called": ", ".join(sorted(need - set(used))) or "-"})
display(pd.DataFrame(rows))
print("`unnecessary` is a description problem. `never_called` is usually a worse one:")
print("the agent answered a question it did not have the information to answer.")

## D. The economics of agency · 15 min

Two effects compound.

**Reliability.** If each step succeeds with probability *p*, an *n*-step task succeeds
with probability *pⁿ*. At *p* = 0.95, ten steps succeed 60 % of the time.

**Cost.** The whole trace is re-sent at every step: Session 2's *n(n+1)/2* formula, with
about 600 tokens per step instead of ~120. Twelve steps is 46 800 input tokens, about
35× a single call.

The cell below computes both from your measured numbers.


In [ ]:
# Two computations on your Part C numbers.
# 1. p^n: probability that an n-step chain succeeds when each step succeeds with probability p.
# 2. c * n(n+1)/2: total input tokens when a trace of c tokens per step is re-sent at every step.

print("RELIABILITY COMPOUNDS")
print(f"{'steps':>6} | " + " | ".join(f"p={p:.2f}" for p in (0.90, 0.95, 0.98, 0.99)))
for n in (3, 5, 8, 10, 12, 20):
    print(f"{n:>6} | " + " | ".join(f"{p ** n:>6.0%}" for p in (0.90, 0.95, 0.98, 0.99)))

print("\nTOKENS GROW QUADRATICALLY")
c_measured = C.in_tokens.sum() / max(1, C.steps.sum())
print(f"your measured tokens per step (c) : {c_measured:.0f}")
for n in (3, 6, 12, 20):
    tot = c_measured * n * (n + 1) / 2
    print(f"  {n:>2} steps -> {tot:>9,.0f} input tokens  "
          f"= {tot * CFG['price_in'] / 1e6:.5f} EUR  "
          f"= {tot / c_measured:.0f}x a single call")



In [ ]:
# Extrapolation from the measured mean cost per request to 400 requests per day.
print("\nYOUR AGENT, AT VELORA'S VOLUME (400 requests/day)")
eur = C.cost_eur.mean()
print(f"  mean steps        : {C.steps.mean():.1f}")
print(f"  mean EUR/request  : {eur:.5f}")
print(f"  per day           : {eur * 400:.2f} EUR")
print(f"  per year          : {eur * 400 * 365:,.0f} EUR      <- for your report")
print(f"  p95 latency       : {C.latency_s.quantile(0.95):.2f} s "
      f"(n={len(C)}, so this is close to the maximum — say so)")
# Baseline: one call of c tokens in and ~200 tokens out, with no loop.
single = c_measured * CFG["price_in"] / 1e6 + 200 * CFG["price_out"] / 1e6
print(f"\n  a single non-agentic call would cost ~{single:.6f} EUR, so your agent is "
      f"{eur / single:.0f}x that.")

**The takeaway.** A feature that looks free in a demo would cost substantially more at four hundred requests a day. Nothing is broken: the trace is re-sent at
every step, and the model chose how many steps there would be.

`max_steps` caps the bill, not the correctness. A partial answer looks like a complete
one to anyone who only reads the output.


## E. Reading the traces · 20 min

Read the steps, not the summary table.

Failure taxonomy. Each code points to a different fix.

| code | failure |
|---|---|
| **F-TOOL** | wrong tool for the question. The description did not set it apart from a neighbour. |
| **F-ARG** | right tool, wrong or invented argument. |
| **F-LOOP** | the same call repeated with no new information. |
| **F-PROP** | an early error propagates. Step 2 is wrong, steps 3–6 build on it. |
| **F-ARITH** | arithmetic done in the model's head instead of with `calculate`. |
| **F-FAB** | a fact in the final answer that appears in no tool result. |
| **F-STOP** | stopped too early, or hit the cap and returned a partial answer as complete. |

**F-FAB is the one to hunt.** One question asks for something that exists nowhere in the
system; the agent should say so. Find what your agent did instead, and quote the line.


In [ ]:
# Trace printer: one block per step with latency, tokens, the model's text, each tool
# call with its (truncated) result, and the final answer if any.
def show_trace(qid, runs=None, max_chars=420):
    """Print one trace, step by step, as it will be read in your report."""
    r = (runs or runs_c)[qid]
    print("=" * 78)
    print(f"{qid}  steps={r['steps']}  floor={r.get('min_steps')}  stop={r['stop_reason']}  "
          f"eur={r['cost_eur']:.5f}")
    print("=" * 78)
    for s in r["trace"]:
        print(f"\n--- step {s['step']}  ({s['latency_s']}s, "
              f"{s['in_tokens']} in / {s['out_tokens']} out)")
        if s["thought"].strip():
            print(f"  THOUGHT: {s['thought'][:300]}")
        for c in s["calls"]:
            print(f"  CALL   : {c['tool']}({json.dumps(c['args'], ensure_ascii=False)[:140]})")
            print(f"  RESULT : {c['result'][:max_chars]}")
        if s.get("final"):
            print(f"  FINAL  : {s['final'][:600]}")

show_trace(REFERENCE[0]["id"])   # start here, then work through the rest

**Observations.** Everything the agent could legitimately know for a question is the
concatenation of its tool results. A claim in the final answer that is not in this text
was invented.


In [ ]:
# observations(qid) concatenates every tool result the agent received for one question.
# This text is the complete set of facts the agent could legitimately use.
# Everything your agent could have seen for one question. If a fact in the final
# answer is not in this text, the agent made it up.
def observations(qid, runs=None):
    r = (runs or runs_c)[qid]
    return "\n".join(c["result"] for s in r["trace"] for c in s["calls"])

for qid in [q["id"] for q in REFERENCE]:
    obs = observations(qid)
    print(f"{qid}: {len(obs)} characters of tool output across "
          f"{sum(len(s['calls']) for s in runs_c[qid]['trace'])} calls")
    print(f"final answer: {runs_c[qid]['answer']}")
    print()

print("Now go through each final answer clause by clause. For each factual claim,")
print("find it in that question's observations. Anything you cannot find is F-FAB.")

### TODO 3 — the trust boundary

Every tool result is appended to `messages` and read by the model at the next step.

| tool | what comes back |
|---|---|
| `query_orders` | your own database, with a schema you wrote |
| `search_kb` | your own documents |
| `calculate` | a number your own code produced |
| `search_web` | text from the open internet, in the same prompt as your instructions |

For each tool, decide what enters the prompt: **raw**, **truncated** (to how much?),
**labelled** (as what?), or **filtered** (of what?). Then say in one sentence which tool
worries you and why.


In [ ]:
# TODO 3 workspace. One decision per tool: how its output enters the prompt.
TRUST_BOUNDARY = {
    "query_orders": ...,   # <- "raw" / "truncated to N" / "labelled" / "filtered"
    "search_kb":    ...,
    "calculate":    ...,
    "search_web":   ...,
}
WHICH_WORRIES_ME = ...     # <- one sentence

assert set(TRUST_BOUNDARY) == {"query_orders", "search_kb", "calculate", "search_web"}
assert all(isinstance(v, str) and len(v) > 20 for v in TRUST_BOUNDARY.values())
assert isinstance(WHICH_WORRIES_ME, str) and len(WHICH_WORRIES_ME) > 60
for k, v in TRUST_BOUNDARY.items():
    print(f"{k:14s} {v}")
print(f"\n{WHICH_WORRIES_ME}")

### TODO 4 — the failure contract

Write what your system does when a tool fails, as a rule your **code** applies. Four
cases; for each, what the agent is told and what the caller receives:

1. a tool times out;
2. a tool returns an **empty result** (this is information, not a failure);
3. a tool returns an error the model cannot act on;
4. the step budget runs out.

Case 2 is the hard one. An agent that reads "no results" as "the search failed"
rephrases the query three times and then answers anyway.

The cell then tests your contract by replaying Q1 with a failing search tool.


In [ ]:
# TODO 4 workspace, then a test: Q1 with search_web failing (timeout, empty, timeout, timeout).
# The contract is text here; the test shows whether the current code already behaves that way.
FAILURE_CONTRACT = {
    "timeout": ...,      # <- what the agent is told, and what the caller receives
    "empty":   ...,
    "error":   ...,
    "budget":  ...,
}

assert set(FAILURE_CONTRACT) == {"timeout", "empty", "error", "budget"}
assert all(isinstance(v, str) and len(v) > 50 for v in FAILURE_CONTRACT.values())
for k, v in FAILURE_CONTRACT.items():
    print(f"{k:8s} {v}")

_saved = SEARCH_MODE
SEARCH_MODE = "replay_fail"
FAIL_PLAN.clear(); FAIL_PLAN.extend(["timeout", "empty", "timeout", "timeout"])
print("\n" + "=" * 78 + "\nQ1 with a failing search tool:\n")
_a, _s = run_agent(BY_QID["Q1"]["question"], TOOLS_V2, SYSTEM_V2,
                   max_steps=STEP_BUDGET, verbose=False)
SEARCH_MODE = _saved
print(f"steps={_s['steps']}  stop={_s['stop_reason']}")
print(f"answer: {_a}")
print("\nDid it do what your contract says? If not, the contract is aspiration, not code.")

### Annotate the traces

One row per (question, step) that went wrong. Precision scores: the exact step, the exact
evidence, the exact cause.


In [ ]:
# One dict per (question, step) that went wrong. evidence quotes the trace; cause names
# the description, prompt or code gap behind it.
# ------------------------------------------------------- annotate your five traces
# One row per (question, step) that went wrong. Codes: F-TOOL F-ARG F-LOOP F-PROP
# F-ARITH F-FAB F-STOP. Be specific: the exact step and the exact reason.
#
# "the agent looped" is worth one mark out of six.
# "step 4: search_web chosen for an orders question because its description said
#  'any information'" is worth six.

ANNOTATIONS = [
    # {"qid": "Qx", "step": 3, "code": "F-FAB",
    #  "evidence": "the final answer states a fact that appears in no tool result",
    #  "cause": "no instruction to refuse, and nothing in the loop checks that a claim
    #            appears in an observation"},
]

if ANNOTATIONS:
    ann = pd.DataFrame(ANNOTATIONS)
    display(ann)
    print("\nby code:", dict(ann.code.value_counts()))
else:
    print("No annotations yet. Go back to show_trace() and read.")

### The trace is not the reasoning

The `Thought:` lines are generated text: a plausible story, sometimes written after the
fact, sometimes contradicted by the tool call that follows.

> **You audit actions. You do not audit intentions.**

What you can audit is the **log**: tool names, arguments, results, timestamps. These are
facts your own code recorded. The trace helps you debug. The log is evidence.


In [ ]:
# The audit log from Part C: calls per tool and how many returned without an error.
# TOOL_LOG was cleared before Part C, so this covers the five reference runs only.
# The audit log: what your code recorded, not what the model said.
log = pd.DataFrame(TOOL_LOG)
if len(log):
    display(log.groupby("tool").agg(calls=("tool", "size"), ok=("ok", "sum")))
    print(f"\n{len(log)} tool invocations recorded across this session.")
    print("None of these lines is generated text. Every one is something your code did.")
else:
    print("TOOL_LOG is empty — run part C first.")

## F. Bonus

Pick one. A measured negative result beats an unmeasured feature.

- **A write tool behind a human gate.** Add `issue_credit_note`, guarded by a predicate
  you write. Then ask something that does not need a credit note, and see if the agent
  reaches for it anyway.
- **Cheaper descriptions.** Cut your TODO 1 text in half and re-run Part B. When do step
  counts start climbing again?
- **A fifth tool that overlaps a fourth.** Add `search_orders_fulltext` next to
  `query_orders` and measure how often the model picks the wrong one.
- **The stale rate.** Ask for a currency conversion without naming a period. Check the
  `published` field on the page your agent used.
- **The bench against the web.** Run one question twice, once on the recording and once
  on live search, and diff what the agent was able to see. Cell below.


**A write tool.** Every tool so far was read-only. This one changes state (a row in a
ledger), which is why it needs a gate.


In [ ]:
# Bonus: a tool with a side effect. Nothing in run_agent distinguishes it from the
# read-only tools.
# The first tool in this lab that writes something. It leaves a row behind.
LEDGER = []

def issue_credit_note(order_id, amount_eur, reason):
    """Write a credit note to the ledger."""
    LEDGER.append({"order_id": order_id, "amount_eur": amount_eur, "reason": reason,
                   "at": time.strftime("%Y-%m-%dT%H:%M:%S")})
    return {"credit_note_id": f"CN-{len(LEDGER):04d}", "status": "issued"}

print("issue_credit_note defined. The ledger is empty:", LEDGER)

### TODO 5 (optional) — the write gate

A predicate over `(tool_name, args)`, run in the dispatcher, in your code. Return `True`
to allow the call. Decide what must be true of the arguments, what requires a human, and
how the agent is told it was refused.


In [ ]:
# TODO 5 workspace. write_gate() is a predicate over (tool_name, args). guarded_dispatch()
# calls it before any write and returns a refusal dict the model can read.
# The two test calls: a small credit note (expected: allowed) and a large one (expected: refused).
def write_gate(tool_name, args):
    ...

GATE_RATIONALE = ...    # <- one sentence: why this lives in code and not in the prompt

# The write tool joins the dispatch table like any other tool. The gate below is
# the control.
WRITE_DISPATCH = {**DISPATCH, "issue_credit_note": issue_credit_note}

def guarded_dispatch(tool_name, args):
    if tool_name == "issue_credit_note" and not write_gate(tool_name, args):
        return {"error": "refused_by_policy",
                "detail": "this action requires human approval"}
    return WRITE_DISPATCH.get(tool_name, lambda **k: {"error": "unknown_tool"})(**args)

print(guarded_dispatch("issue_credit_note",
                       {"order_id": "VLR-2026-04640", "amount_eur": 40, "reason": "goodwill"}))
print(guarded_dispatch("issue_credit_note",
                       {"order_id": "VLR-2026-04640", "amount_eur": 5000, "reason": "goodwill"}))
print("ledger:", LEDGER)

### The bench against the web

Everything above ran against `search_cache.json`. That is a bench: fixed on purpose, so
that pairs comparing step counts compare their tool descriptions and not their search
luck, and so that the defects Part E asks you to find are there to be found.

It is also not the internet. This cell runs the same question twice and diffs what the
agent could **see**. Switching to `live` costs nothing beyond the key you already have.


In [ ]:
# Same question, run once in cached mode and once in live mode. `seen` stores the answer,
# step count, cost and the concatenated tool outputs for each mode.
# One question, two worlds. Nothing else changes: same tools, same prompt, same budget.
QID = "Q9"          # <- Q1 and Q7 are the other interesting ones: both turn on a rate page

_saved_mode = SEARCH_MODE
_budget = STEP_BUDGET if isinstance(STEP_BUDGET, int) else HARD_CAP
seen = {}

for _mode in ("cached", "live"):
    SEARCH_MODE = _mode
    TOOL_LOG.clear()
    _ans, _s = run_agent(BY_QID[QID]["question"], TOOLS_V2, SYSTEM_V2,
                         max_steps=_budget, verbose=False)
    seen[_mode] = {
        "answer": _ans, "steps": _s["steps"], "cost": _s["cost_eur"],
        "obs": "\n".join(c["result"] for st in _s["trace"] for c in st["calls"]),
    }

SEARCH_MODE = _saved_mode        # put it back, or every cell below changes meaning

for _mode, r in seen.items():
    print("=" * 78)
    print(f"{_mode:7}  steps={r['steps']}  eur={r['cost']:.5f}  obs={len(r['obs'])} chars")
    print(f"         {str(r['answer'])[:320]}")
print("=" * 78)
print(f"search mode is back to {SEARCH_MODE!r}")

Three questions for the report. The third is the one worth writing down.

1. **Did the two runs agree?** If not, which answer would you ship, and on what
   evidence, not which one felt better.
2. **Diff the two observation texts.** What was in the bench that the web did not give
   you, and what was on the web that the bench cannot have?
3. **What does the live run give you no way to check?** You ran it once. Tomorrow it
   returns something else, and the trace filed today no longer reproduces. This is the
   audit-log problem of Part E, arriving from the other direction.


## Deliverable

In groups. Submit:

1. **This notebook, executed** — TODOs filled in, five traces visible — Wednesday 23:59.
2. **A one-page report, by Wednesday 23:59 :**

| | |
|---|---|
| The agent | it runs, and the five reference traces are attached |
| Trace analysis | failures classified with the taxonomy, one exact step cited per class |
| Cost | €/request on your step counts, and the annual figure at 400 requests/day |
| Guardrails | each one justified by something you measured |
| A refusal | one behaviour you would not accept in production, with the trace line |


Indicative evaluation criteria out of 20:

| | |
|---|---|
| The agent runs and five traces are attached | 5 |
| Trace analysis with causes classified | 6 |
| Cost per request computed on your own step count | 4 |
| Guardrails justified by a measurement | 3 |
| One behaviour you would refuse in production, with the trace line | 2 |

"It looped" scores 1 out of 6. "Step 4: `search_web` chosen for a database question
because its description said *any information*" scores 6.

If your search results came from the cache, say so. A latency without a label is not a
measurement.
